In [9]:
"""
Phase 3 — CVE Knowledge Graph Builder
======================================
Builds a unipartite CVE-CVE adjacency matrix by joining through the CPE bridge.

Actual file schemas
-------------------
edges_cve_cpe.csv     : CVE_ID, CPE_ID, Relation
                        (Relation filter: AFFECTS)
edges_cpe_product.csv : CPE_ID, Product_ID, Version, Relation
                        (Relation filter: IS_VERSION_OF)

Join chain
----------
CVE_ID → CPE_ID  (edges_cve_cpe,     keep AFFECTS rows)
CPE_ID → Product_ID + Version  (edges_cpe_product, keep IS_VERSION_OF rows)
Result → CVE_ID, Product_ID, Version

Why we join instead of parsing CPE strings (Trap A fix)
--------------------------------------------------------
edges_cpe_product.csv already has Product_ID ("freerdp:freerdp") and Version
("1.0.0") pre-extracted. We use CPE_ID purely as a join key, never splitting
it. CPE 2.2 vs 2.3 format differences are irrelevant — format only matters
if you parse field positions, which we never do.

Degree-aware capping (Trap B fix)
----------------------------------
When a group exceeds the degree cap we do NOT randomly sample.
We rank the CVE members by their current degree in the partially-built
graph and keep the lowest-degree nodes first. This prioritises nodes that
have few other connections and would otherwise become isolated, minimising
the isolated_pct metric without changing the cap size.

Two edge types
--------------
  Exact-version edges : CVEs sharing Product_ID + Version  (tight homophily)
                        Cap = MAX_VERSION_DEGREE (default 50)
  Product-family edges: CVEs sharing Product_ID only       (soft homophily)
                        Cap = MAX_PRODUCT_DEGREE (default 20)
                        Generic OS/browser products blocked at family level.
                        Exact-version edges for those products still allowed.

Required inputs
---------------
  cve_id_mapping_v4.csv   — CVE_ID, row_index  (Phase 2 output)
  edges_cve_cpe.csv       — CVE_ID, CPE_ID, Relation
  edges_cpe_product.csv   — CPE_ID, Product_ID, Version, Relation

Outputs
-------
  adjacency_v4.npz          — [N,N] sparse CSR, symmetric, no self-loops
  adjacency_selfloop_v4.npz — same + identity  (DOMINANT / CoLA requirement)
  graph_stats_v4.json       — degree stats, edge counts, isolation metrics
"""

import json
import logging
import os
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.sparse as sp

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Exact-version groups larger than this are hub-capped.
# A single software version shared by 50+ CVEs is a genuine hub.
MAX_VERSION_DEGREE = 50

# Product-family groups are softer — cap tighter to prevent OS bloat.
MAX_PRODUCT_DEGREE = 20

# Relation values to keep from each file.
# All other relation types are silently dropped.
CVE_CPE_RELATION     = "AFFECTS"
CPE_PRODUCT_RELATION = "IS_VERSION_OF"

# Product IDs blocked at the family level only.
# These products are so broad that even a capped group of 20 is meaningless.
# NOTE: exact-version edges for these products are still formed —
#       e.g. linux:linux_kernel + 5.15.0 is specific and useful.
BLOCKED_PRODUCT_FAMILIES = {
    "microsoft:windows",
    "linux:linux_kernel",
    "apple:macos",
    "apple:mac_os_x",
    "google:chrome",
    "mozilla:firefox",
    "oracle:jdk",
    "oracle:jre",
    "apache:http_server",
    "microsoft:internet_explorer",
    "adobe:flash_player",
    "adobe:acrobat_reader",
}


In [10]:


# ===========================================================================
# GRAPH BUILDER
# ===========================================================================

class CVEGraphBuilder:
    """
    Builds the CVE knowledge graph by joining CVE → CPE → Product.

    The CPE string is used only as a join key, never parsed for fields.
    Product_ID and Version come from the pre-parsed edges_cpe_product.csv.
    """

    def __init__(
        self,
        mapping_path:     str = "cve_id_mapping_v4.csv",
        cve_cpe_path:     str = "edges_cve_cpe.csv",
        cpe_product_path: str = "edges_cpe_product.csv",
    ):
        for p in [mapping_path, cve_cpe_path, cpe_product_path]:
            if not os.path.exists(p):
                raise FileNotFoundError(
                    f"Required file missing: {p}\n"
                    "Run Phase 2 first (cve_id_mapping_v4.csv) and ensure "
                    "edge files are in the working directory."
                )

        self.cve_cpe_path     = cve_cpe_path
        self.cpe_product_path = cpe_product_path

        mapping         = pd.read_csv(mapping_path)
        self.cve_to_idx = dict(zip(mapping["CVE_ID"], mapping["row_index"]))
        self.N          = len(mapping)

        log.info(f"Graph node count : N = {self.N:,} CVEs")

        # Running degree counter — updated as edges are added.
        # Used by degree-aware capping to prefer low-degree nodes.
        self._degree = np.zeros(self.N, dtype=np.int32)

    # -----------------------------------------------------------------------
    # Step 1 — Load and join edge files
    # -----------------------------------------------------------------------

    def _load_cve_product_table(self) -> pd.DataFrame:
        """
        Joins edges_cve_cpe and edges_cpe_product on CPE_ID.
        Returns a DataFrame with columns: CVE_ID, cve_idx, Product_ID, Version.

        CPE strings are used ONLY as join keys — never split or parsed.
        Product_ID and Version come from the pre-parsed product edge file.
        """
        log.info("─" * 60)
        log.info("STEP 1 — Loading and joining edge files")
        log.info("─" * 60)

        # ── Load CVE → CPE edges ─────────────────────────────────────────
        cve_cpe = pd.read_csv(
            self.cve_cpe_path,
            engine="python",
            quotechar='"',
            on_bad_lines="warn",
        )
        log.info(f"edges_cve_cpe     : {len(cve_cpe):,} rows loaded")

        _check_columns(cve_cpe, ["CVE_ID", "CPE_ID"], self.cve_cpe_path)

        # Filter to AFFECTS relation if column exists
        if "Relation" in cve_cpe.columns:
            before   = len(cve_cpe)
            cve_cpe  = cve_cpe[cve_cpe["Relation"] == CVE_CPE_RELATION].copy()
            log.info(
                f"  After AFFECTS filter    : {len(cve_cpe):,} rows "
                f"({before - len(cve_cpe):,} non-AFFECTS dropped)"
            )

        # Keep only CVEs that are in our Phase 2 mapping
        cve_cpe  = cve_cpe[cve_cpe["CVE_ID"].isin(self.cve_to_idx)].copy()
        log.info(f"  After mapping filter    : {len(cve_cpe):,} rows")

        # ── Load CPE → Product edges ──────────────────────────────────────
        cpe_prod = pd.read_csv(
            self.cpe_product_path,
            engine="python",
            quotechar='"',
            on_bad_lines="warn",
        )
        log.info(f"edges_cpe_product : {len(cpe_prod):,} rows loaded")

        _check_columns(
            cpe_prod, ["CPE_ID", "Product_ID", "Version"], self.cpe_product_path
        )

        # Filter to IS_VERSION_OF relation if column exists
        if "Relation" in cpe_prod.columns:
            before   = len(cpe_prod)
            cpe_prod = cpe_prod[
                cpe_prod["Relation"] == CPE_PRODUCT_RELATION
            ].copy()
            log.info(
                f"  After IS_VERSION_OF     : {len(cpe_prod):,} rows "
                f"({before - len(cpe_prod):,} other relations dropped)"
            )

        # Deduplicate CPE_ID → Product_ID + Version mappings
        cpe_prod = cpe_prod[["CPE_ID", "Product_ID", "Version"]].drop_duplicates()

        # ── Join on CPE_ID ────────────────────────────────────────────────
        # Inner join: only CPE_IDs present in both tables are kept.
        # CPE string format never matters — it is only a join key here.
        merged = cve_cpe[["CVE_ID", "CPE_ID"]].merge(
            cpe_prod, on="CPE_ID", how="inner"
        )
        log.info(f"After join        : {len(merged):,} rows")
        log.info(
            f"  Unique CVEs     : {merged['CVE_ID'].nunique():,}  "
            f"| Unique products : {merged['Product_ID'].nunique():,}  "
            f"| Unique versions : {merged['Version'].nunique():,}"
        )

        # Attach row index
        merged["cve_idx"] = merged["CVE_ID"].map(self.cve_to_idx).astype(int)

        # Normalise strings
        merged["Product_ID"] = merged["Product_ID"].str.strip().str.lower()
        merged["Version"]    = merged["Version"].str.strip().str.lower()

        # Drop rows with null Product_ID or Version
        before  = len(merged)
        merged  = merged.dropna(subset=["Product_ID"])
        merged["Version"] = merged["Version"].fillna("")
        merged["has_version"] = merged["Version"].str.strip().isin(["","-","*"]) == False
        dropped = before - len(merged)
        if dropped:
            log.warning(f"Dropped {dropped:,} rows with null Product_ID or Version")

        return merged

    # -----------------------------------------------------------------------
    # Step 2 — Build group dictionaries
    # -----------------------------------------------------------------------
    def _build_groups(self, df: pd.DataFrame) -> tuple:
        """
        From the joined CVE-product table, build two group dicts:
          exact_groups[product:version] = [cve_idx, ...]
          family_groups[product]        = [cve_idx, ...]
        """
        log.info("─" * 60)
        log.info("STEP 2 — Building group dictionaries (Vectorized)")
        log.info("─" * 60)

        exact_groups:  dict[str, list] = defaultdict(list)
        family_groups: dict[str, list] = defaultdict(list)

        # Exact-version groups — only rows with a real version (Activates Null Rescue)
        df_versioned = df[df["has_version"]].copy()
        df_versioned["exact_key"] = df_versioned["Product_ID"] + "::" + df_versioned["Version"]

        for key, grp in df_versioned.groupby("exact_key"):
            exact_groups[key] = grp["cve_idx"].tolist()

        # Product-family groups — exclude blocked products
        df_family = df[~df["Product_ID"].isin(BLOCKED_PRODUCT_FAMILIES)].copy()

        for pid, grp in df_family.groupby("Product_ID"):
            family_groups[pid] = grp["cve_idx"].tolist()

        log.info(f"  Exact-version groups : {len(exact_groups):,}")
        log.info(f"  Product-family groups: {len(family_groups):,}")
        log.info(f"  Blocked at family    : {len(BLOCKED_PRODUCT_FAMILIES)} products")

        return exact_groups, family_groups

    # -----------------------------------------------------------------------
    # Step 3 — Degree-aware capping and pair generation
    # -----------------------------------------------------------------------

    def _degree_aware_cap(
        self, members: list, cap: int
    ) -> list:
        """
        When a group has more than `cap` members, select `cap` to keep.

        Strategy: prefer nodes with the lowest current degree.
        Low-degree nodes have the fewest connections elsewhere — dropping them
        would most likely make them isolated. High-degree nodes have other edges
        to fall back on and can afford to lose this connection.

        This minimises isolated_pct without changing the cap value.
        """
        if len(members) <= cap:
            return members
        members   = list(set(members))          # remove duplicates
        degrees   = self._degree[members]       # current degrees of candidates
        order     = np.argsort(degrees)         # ascending: lowest degree first
        kept_idx  = order[:cap]
        return [members[i] for i in kept_idx]

    def _groups_to_pairs(
        self,
        groups: dict,
        cap:    int,
        label:  str,
    ) -> list:
        pairs         = []
        seen          = set()   # track already-added pairs for accurate degree
        n_capped      = 0
        n_skipped     = 0
        n_groups_used = 0

        for key, raw_members in groups.items():
            members = list(set(raw_members))
            if len(members) < 2:
                n_skipped += 1
                continue

            if len(members) > cap:
                members = self._degree_aware_cap(members, cap)
                n_capped += 1

            new_pairs = list(combinations(sorted(members), 2))
            n_groups_used += 1

            for i, j in new_pairs:
                edge_key = (min(i,j), max(i,j))
                if edge_key not in seen:
                    seen.add(edge_key)
                    pairs.append((i, j))
                    # Only increment for genuinely new edges
                    self._degree[i] += 1
                    self._degree[j] += 1

        log.info(
            f"  {label:<22}: {n_groups_used:>6,} groups  "
            f"| {n_capped:>5,} capped  "
            f"| {n_skipped:>5,} singletons  "
            f"| {len(pairs):>8,} unique pairs"
        )
        return pairs

    # -----------------------------------------------------------------------
    # Step 4 — Build sparse CSR matrix
    # -----------------------------------------------------------------------

    def _pairs_to_csr(self, pairs: list) -> sp.csr_matrix:
        """
        Deduplicate (i, j) pairs, remove self-loops, enforce symmetry,
        return a float32 CSR matrix of shape [N, N].
        """
        if not pairs:
            log.warning("No edge pairs generated — empty adjacency matrix.")
            return sp.csr_matrix((self.N, self.N), dtype=np.float32)

        seen       = set()
        rows, cols = [], []

        for i, j in pairs:
            if i == j:
                continue
            key = (min(i, j), max(i, j))
            if key in seen:
                continue
            seen.add(key)
            rows += [i, j]
            cols += [j, i]

        data = np.ones(len(rows), dtype=np.float32)
        return sp.csr_matrix(
            (data, (rows, cols)),
            shape=(self.N, self.N),
            dtype=np.float32,
        )
    # -----------------------------------------------------------------------
    # Step 5 — CWE Rescue Pass
    # -----------------------------------------------------------------------
    def _cwe_rescue_pass(self, adj, cwe_edges_path="edges_cve_cwe.csv"):

        import random

        random.seed(42)

        FAKE_CWES    = {"NVD-CWE-noinfo", "NVD-CWE-Other"}
        CWE_RESCUE_K = 15   # max rescue edges per isolated node

        degrees      = np.asarray(adj.sum(axis=1)).flatten()
        
        isolated_set = set(np.where(degrees == 0)[0])
        log.info(f"  Isolated before CWE rescue : {len(isolated_set):,}")

        if not isolated_set:
            log.info("  No isolated nodes — CWE rescue pass skipped")
            return adj

        edges_cwe = pd.read_csv(cwe_edges_path)
        edges_cwe = edges_cwe[~edges_cwe["Weakness_CWE"].isin(FAKE_CWES)]

        # Track rescue edges per node — cap at K
        node_rescue_count = defaultdict(int)
        rescue_pairs      = []
        seen              = set()
        rescued           = set()

        for cwe, grp in edges_cwe.groupby("Weakness_CWE"):
            # Only isolated CVEs in this CWE group
            cve_indices = [
                self.cve_to_idx[c]
                for c in grp["CVE_ID"]
                if c in self.cve_to_idx
                and self.cve_to_idx[c] in isolated_set
            ]
            if len(cve_indices) < 2:
                continue

            # For each isolated node in this CWE group, sample K neighbors
            # from the same group — do NOT form full clique
            cve_indices_set = list(set(cve_indices))
            random.shuffle(cve_indices_set)

            for node in cve_indices_set:
                if node_rescue_count[node] >= CWE_RESCUE_K:
                    continue   # node already has enough rescue edges

                candidates = [
                    n for n in cve_indices_set
                    if n != node and node_rescue_count[n] < CWE_RESCUE_K
                ]
                slots   = CWE_RESCUE_K - node_rescue_count[node]
                sampled = random.sample(candidates, min(slots, len(candidates)))

                for nb in sampled:
                    key = (min(node, nb), max(node, nb))
                    if key not in seen:
                        seen.add(key)
                        rescue_pairs.append((node, nb))
                        node_rescue_count[node] += 1
                        node_rescue_count[nb]   += 1
                        rescued.add(node)
                        rescued.add(nb)

        if not rescue_pairs:
            log.info("  CWE rescue found no valid pairs")
            return adj

        rows = [i for i, j in rescue_pairs] + [j for i, j in rescue_pairs]
        cols = [j for i, j in rescue_pairs] + [i for i, j in rescue_pairs]
        data = np.ones(len(rows), dtype=np.float32)

        rescue_adj  = sp.csr_matrix((data, (rows, cols)),
                                 shape=(self.N, self.N), dtype=np.float32)
        adj_rescued = adj + rescue_adj
        adj_rescued.data[:] = 1.0

        new_degrees    = np.asarray(adj_rescued.sum(axis=1)).flatten()
        still_isolated = int((new_degrees == 0).sum())

        log.info(f"  CWE rescue K         : {CWE_RESCUE_K}")
        log.info(f"  CWE rescue pairs     : {len(rescue_pairs):,}")
        log.info(f"  Nodes rescued        : {len(rescued):,}")
        log.info(f"  Still isolated       : {still_isolated:,}  "
                 f"({still_isolated / self.N * 100:.1f}%)")

        return adj_rescued


    # -----------------------------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------------------------

    def _diagnostics(
        self,
        adj:           sp.csr_matrix,
        exact_pairs:   int,
        family_pairs:  int,
    ) -> dict:
        degrees  = np.asarray(adj.sum(axis=1)).flatten()

        self.plot_degree_distribution(degrees)
        
        isolated = int((degrees == 0).sum())
        density  = float(adj.nnz / (self.N * self.N))

        log.info("─" * 60)
        log.info("GRAPH DIAGNOSTICS")
        log.info("─" * 60)
        log.info(f"  Nodes              : {self.N:,}")
        log.info(f"  Edges (undirected) : {adj.nnz // 2:,}")
        log.info(f"    from exact-version pairs : {exact_pairs:,}")
        log.info(f"    from product-family pairs: {family_pairs:,}")
        log.info(f"  Isolated nodes     : {isolated:,}  ({isolated / self.N * 100:.1f}%)")
        log.info(f"  Degree — mean      : {degrees.mean():.2f}")
        log.info(f"  Degree — median    : {np.median(degrees):.0f}")
        log.info(f"  Degree — p95       : {np.percentile(degrees, 95):.0f}")
        log.info(f"  Degree — max       : {int(degrees.max())}")
        log.info(f"  Density            : {density:.8f}")

        # --- HUB HUNTER LOGIC ---
        if int(degrees.max()) > 2000:
            log.warning("⚠️ MASSIVE HUB DETECTED! Printing top 5 highest degree CVEs...")
            top_nodes = np.argsort(degrees)[::-1][:5]
            idx_to_cve = {v: k for k, v in self.cve_to_idx.items()}
            for node in top_nodes:
                log.warning(f"    Degree: {int(degrees[node]):<6} | CVE: {idx_to_cve.get(node, '?')}")
            log.warning("→ Look up these CVEs in your products table and add the offending software to BLOCKED_PRODUCT_FAMILIES.")

        # Actionable warnings
        iso_pct = isolated / self.N * 100
        if iso_pct > 30:
            log.warning(
                f"Isolated nodes at {iso_pct:.1f}% — above 30% threshold.\n"
                f"  → Consider raising MAX_VERSION_DEGREE from "
                f"{MAX_VERSION_DEGREE} to 100 or 150.\n"
                f"  → These isolated CVEs will receive no neighbourhood "
                f"signal from the GNN."
            )
        elif iso_pct > 15:
            log.warning(
                f"Isolated nodes at {iso_pct:.1f}% — monitor closely.\n"
                f"  → If GNN performance is low, try MAX_VERSION_DEGREE=75."
            )
        else:
            log.info(f"  Isolated node pct  : {iso_pct:.1f}%  ✓ (below 15%)")

        if int(degrees.max()) > 2000:
            log.warning(
                f"Max degree = {int(degrees.max())} — hub still present.\n"
                f"  → Add its Product_ID to BLOCKED_PRODUCT_FAMILIES."
            )

        if density < 1e-5:
            log.warning(
                "Graph is very sparse. GNNs may underperform.\n"
                "  → Consider raising MAX_VERSION_DEGREE."
            )

        return {
            "num_nodes":          self.N,
            "num_edges":          int(adj.nnz // 2),
            "exact_version_pairs":  exact_pairs,
            "product_family_pairs": family_pairs,
            "isolated_nodes":     isolated,
            "isolated_pct":       round(iso_pct, 2),
            "density":            round(density, 10),
            "degree_mean":        round(float(degrees.mean()), 3),
            "degree_median":      int(np.median(degrees)),
            "degree_p95":         float(np.percentile(degrees, 95)),
            "degree_max":         int(degrees.max()),
            "max_version_degree": MAX_VERSION_DEGREE,
            "max_product_degree": MAX_PRODUCT_DEGREE,
        }
    
    def plot_degree_distribution(self, degrees: np.ndarray, save_path: str = "fig_degree_distribution_v4.pdf"):
        """
        Plots the log-log degree distribution of the graph to visually prove
        the effectiveness of the hub-capping strategy.
        """
        import matplotlib
        matplotlib.use("Agg")  # Safe for headless/script execution
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(8, 5))

        # Filter out isolated nodes (degree 0) for the log-log scale
        active_degrees = degrees[degrees > 0]

        if len(active_degrees) == 0:
            log.warning("Graph has no edges. Skipping degree plot.")
            return

        # Create logarithmically spaced bins for a clean log-log histogram
        max_deg = max(active_degrees.max(), 1)
        bins = np.logspace(0, np.log10(max_deg + 1), num=50)
        
        ax.hist(active_degrees, bins=bins, color="#3B82F6", alpha=0.8, edgecolor="none")

        # Set Log-Log Scale (Standard for Knowledge Graphs)
        ax.set_xscale("log")
        ax.set_yscale("log")

        # Draw the Capping Thresholds to prove they worked
        ax.axvline(MAX_PRODUCT_DEGREE, color="#10B981", linestyle="--", linewidth=1.5, label=f"Prod Cap ({MAX_PRODUCT_DEGREE})")
        ax.axvline(MAX_VERSION_DEGREE, color="#EF4444", linestyle="--", linewidth=1.5, label=f"Ver Cap ({MAX_VERSION_DEGREE})")
        
        median_deg = int(np.median(active_degrees))
        ax.axvline(median_deg, color="black", linestyle=":", linewidth=1.5, label=f"Median ({median_deg})")

        ax.set_title("CVE Graph Degree Distribution (Log-Log)", fontsize=12, fontweight="bold")
        ax.set_xlabel("Node Degree (Number of Edges)", fontsize=11)
        ax.set_ylabel("Frequency (Number of Nodes)", fontsize=11)
        
        # Format X-axis ticks to show actual numbers instead of 10^1, 10^2
        ax.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
        
        ax.legend(loc="upper right", fontsize=10)
        ax.grid(alpha=0.2, which="both", linestyle="--")

        fig.tight_layout()
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        log.info(f"Saved graph visualization: {save_path}")

    # -----------------------------------------------------------------------
    # Public entry point
    # -----------------------------------------------------------------------

    def build(self) -> dict:
        """
        Full pipeline: load → join → group → cap → pair → assemble → save.
        Returns stats dict.
        """
        log.info("=" * 60)
        log.info("PHASE 3 — CVE KNOWLEDGE GRAPH BUILDER")
        log.info("=" * 60)
        log.info(f"  MAX_VERSION_DEGREE : {MAX_VERSION_DEGREE}")
        log.info(f"  MAX_PRODUCT_DEGREE : {MAX_PRODUCT_DEGREE}")
        log.info(f"  Capping strategy   : degree-aware (not random)")
        log.info(f"  CPE parsing        : NONE (join-key only — Trap A bypassed)")

        # Step 1 — join
        df = self._load_cve_product_table()

        # Step 2 — build groups
        exact_groups, family_groups = self._build_groups(df)

        # Step 3 — pairs  (exact-version first so degree state is populated
        # before family-level capping — nodes already well-connected from
        # exact edges will be deprioritised in family capping)
        log.info("─" * 60)
        log.info("STEP 3 — Generating edge pairs (degree-aware capping)")
        log.info("─" * 60)

        exact_pairs  = self._groups_to_pairs(
            exact_groups, MAX_VERSION_DEGREE, "Exact-version"
        )
        family_pairs = self._groups_to_pairs(
            family_groups, MAX_PRODUCT_DEGREE, "Product-family"
        )
        all_pairs = exact_pairs + family_pairs

        log.info(f"  Total raw pairs    : {len(all_pairs):,}")

        # Step 4 — assemble
        log.info("─" * 60)
        log.info("STEP 4 — Assembling sparse adjacency matrix")
        log.info("─" * 60)

        adj    = self._pairs_to_csr(all_pairs)
        
        diff = adj - adj.T
        assert diff.nnz == 0, f"Adjacency matrix is not symmetric — {diff.nnz} asymmetric entries"
        
        adj = self._cwe_rescue_pass(adj, cwe_edges_path="edges_cve_cwe.csv")

        adj_sl = adj + sp.eye(self.N, dtype=np.float32, format="csr")

        stats = self._diagnostics(adj, len(exact_pairs), len(family_pairs))

        # Save
        sp.save_npz("adjacency_v4.npz",         adj)
        sp.save_npz("adjacency_selfloop_v4.npz", adj_sl)

        with open("graph_stats_v4.json", "w") as f:
            json.dump(stats, f, indent=2)

        log.info("=" * 60)
        log.info("PHASE 3 COMPLETE")
        log.info("=" * 60)
        log.info("  adjacency_v4.npz          → no self-loops (reference)")
        log.info("  adjacency_selfloop_v4.npz → self-loops (DOMINANT/CoLA)")
        log.info("  graph_stats_v4.json       → degree stats")
        log.info("")
        log.info("  Next: run phase4_gnn_ensemble.py")

        return stats




In [11]:

# ===========================================================================
# HELPERS
# ===========================================================================

def _check_columns(df: pd.DataFrame, required: list, path: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"File {path} is missing required columns: {missing}\n"
            f"Found columns: {list(df.columns)}\n"
            f"Check your CSV header row."
        )


In [12]:

# ===========================================================================
# MAIN
# ===========================================================================

def main():
    builder = CVEGraphBuilder(
        mapping_path     = "cve_id_mapping_v4.csv",
        cve_cpe_path     = "edges_cve_cpe.csv",
        cpe_product_path = "edges_cpe_product.csv",
    )

    stats = builder.build()

    print()
    print("=" * 60)
    print("GRAPH SUMMARY")
    print("=" * 60)
    for key, val in stats.items():
        print(f"  {key:<26}: {val}")
    print("=" * 60)
    print()
    print("  Feed adjacency_selfloop_v4.npz → phase4_gnn_ensemble.py")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        log.error(f"Phase 3 failed: {exc}")
        raise


19:42:20  INFO      Graph node count : N = 201,751 CVEs
19:42:20  INFO      ============================================================
19:42:20  INFO      PHASE 3 — CVE KNOWLEDGE GRAPH BUILDER
19:42:20  INFO      ============================================================
19:42:20  INFO        MAX_VERSION_DEGREE : 50
19:42:20  INFO        MAX_PRODUCT_DEGREE : 20
19:42:20  INFO        Capping strategy   : degree-aware (not random)
19:42:20  INFO        CPE parsing        : NONE (join-key only — Trap A bypassed)
19:42:20  INFO      ────────────────────────────────────────────────────────────
19:42:20  INFO      STEP 1 — Loading and joining edge files
19:42:20  INFO      ────────────────────────────────────────────────────────────


19:42:24  INFO      edges_cve_cpe     : 1,374,447 rows loaded
19:42:25  INFO        After AFFECTS filter    : 1,374,447 rows (0 non-AFFECTS dropped)
19:42:25  INFO        After mapping filter    : 1,374,447 rows
19:42:26  INFO      edges_cpe_product : 220,433 rows loaded
19:42:26  INFO        After IS_VERSION_OF     : 220,433 rows (0 other relations dropped)
19:42:27  INFO      After join        : 1,374,447 rows
19:42:27  INFO        Unique CVEs     : 201,751  | Unique products : 89,008  | Unique versions : 37,790
19:42:29  INFO      ────────────────────────────────────────────────────────────
19:42:29  INFO      STEP 2 — Building group dictionaries (Vectorized)
19:42:29  INFO      ────────────────────────────────────────────────────────────
19:42:38  INFO        Exact-version groups : 107,991
19:42:38  INFO        Product-family groups: 88,996
19:42:38  INFO        Blocked at family    : 12 products
19:42:38  INFO      ────────────────────────────────────────────────────────────
19:42


GRAPH SUMMARY
  num_nodes                 : 201751
  num_edges                 : 1982397
  exact_version_pairs       : 959054
  product_family_pairs      : 738047
  isolated_nodes            : 7340
  isolated_pct              : 3.64
  density                   : 9.74068e-05
  degree_mean               : 19.652
  degree_median             : 15
  degree_p95                : 62.0
  degree_max                : 829
  max_version_degree        : 50
  max_product_degree        : 20

  Feed adjacency_selfloop_v4.npz → phase4_gnn_ensemble.py


In [13]:
import scipy.sparse as sp
import numpy as np
import pandas as pd

adj     = sp.load_npz("adjacency_v4.npz")
mapping = pd.read_csv("cve_id_mapping_v4.csv")
idx_to_cve = dict(zip(mapping["row_index"], mapping["CVE_ID"]))

degrees  = np.asarray(adj.sum(axis=1)).flatten()
top10    = np.argsort(degrees)[::-1][:10]

print("Top 10 highest-degree nodes:")
for rank, node in enumerate(top10, 1):
    print(f"  {rank:>2}. degree={degrees[node]:.0f}  CVE={idx_to_cve.get(node, '?')}")

Top 10 highest-degree nodes:
   1. degree=829  CVE=CVE-2019-10219
   2. degree=645  CVE=CVE-2019-11358
   3. degree=501  CVE=CVE-2021-2351
   4. degree=499  CVE=CVE-2021-45105
   5. degree=487  CVE=CVE-2023-44487
   6. degree=455  CVE=CVE-2014-6271
   7. degree=416  CVE=CVE-2021-4104
   8. degree=411  CVE=CVE-2014-7169
   9. degree=395  CVE=CVE-2017-5645
  10. degree=380  CVE=CVE-2021-44228
